# Roteamento Multimodal com A\* (Walk + Drive)

Problema: encontrar o melhor ponto de embarque **P** (a até **X metros** de caminhada de **A**)
para minimizar distância ou tempo total até **B**, usando o algoritmo **A\*** com heurística geográfica.

- **A**: Museu de Morfologia (UFRN)
- **B**: Assaí Atacadista (Natal/RN)
- Dois grafos: rede pedestre (A → P) e rede de carro (P → B)


In [1]:
!pip install osmnx networkx folium matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.9 MB/s eta 0:00:00


In [2]:
PONTO_A = {"nome": "Museu de Morfologia", "lat": -5.841777929488095, "lon": -35.203041910059596}
PONTO_B = {"nome": "Assai Atacadista",   "lat": -5.858685640917017, "lon": -35.195926594143344}

X = 500  # distância máxima de caminhada em metros

print(f"A: {PONTO_A['nome']}")
print(f"B: {PONTO_B['nome']}")
print(f"Caminhada máxima: {X} m")


A: Museu de Morfologia
B: Assai Atacadista
Caminhada máxima: 500 m


In [3]:
import osmnx as ox
import networkx as nx

centro = ((PONTO_A["lat"] + PONTO_B["lat"]) / 2,
          (PONTO_A["lon"] + PONTO_B["lon"]) / 2)

# Grafo de CARRO — trecho P → B
G = ox.graph_from_point(centro, dist=4000, network_type="drive")
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

# Grafo de PEDESTRE — trecho A → P (raio X + margem)
G_walk = ox.graph_from_point(
    (PONTO_A["lat"], PONTO_A["lon"]),
    dist=X + 200,
    network_type="walk"
)

no_A      = ox.distance.nearest_nodes(G,      PONTO_A["lon"], PONTO_A["lat"])
no_A_walk = ox.distance.nearest_nodes(G_walk, PONTO_A["lon"], PONTO_A["lat"])
no_B      = ox.distance.nearest_nodes(G,      PONTO_B["lon"], PONTO_B["lat"])

print(f"Grafo drive — Nós: {G.number_of_nodes()} | Arestas: {G.number_of_edges()}")
print(f"Grafo walk  — Nós: {G_walk.number_of_nodes()} | Arestas: {G_walk.number_of_edges()}")
print(f"Nó A (drive): {no_A} | Nó A (walk): {no_A_walk} | Nó B: {no_B}")


Grafo drive — Nós: 4965 | Arestas: 12359
Grafo walk  — Nós: 938 | Arestas: 2624
Nó A (drive): 501834722 | Nó A (walk): 6956477611 | Nó B: 503296428


In [4]:
import random
import copy

random.seed(42)

# Candidatos P: nós a até X metros de A pela rede pedestre
candidatos = nx.single_source_dijkstra_path_length(
    G_walk, no_A_walk, cutoff=X, weight="length"
)
candidatos = {n: d for n, d in candidatos.items() if n != no_A_walk}

print(f"Candidatos a P (≤ {X} m, rede pedestre): {len(candidatos)} nós")

# Trânsito sintético: fator aleatório aplicado ao grafo de carro
G_tr = copy.deepcopy(G)
for u, v, data in G_tr.edges(data=True):
    data["travel_time_transito"] = data.get("travel_time", 1.0) * random.uniform(1.0, 3.5)

print("Grafo com trânsito sintético pronto.")


Candidatos a P (≤ 500 m, rede pedestre): 82 nós
Grafo com trânsito sintético pronto.


In [5]:
import networkx as nx
import osmnx as ox

def heuristica_geo(u, v, G, peso="length"):
    """
    Heurística admissível baseada na distância geográfica (linha reta).
    Para peso de tempo, converte a distância pela velocidade máxima (120 km/h = 33.3 m/s),
    garantindo que não superestimamos o custo real.
    """
    dist_metros = ox.distance.great_circle(
        G.nodes[u]['y'], G.nodes[u]['x'],
        G.nodes[v]['y'], G.nodes[v]['x']
    )
    if "time" in peso:
        return dist_metros / 33.33  # velocidade máxima → heurística admissível
    return dist_metros

def busca_astar(G, origem, destino, peso="length"):
    """Encontra o caminho mínimo usando A* com heurística geográfica."""
    h = lambda u, v: heuristica_geo(u, v, G, peso=peso)
    custo   = nx.astar_path_length(G, origem, destino, heuristic=h, weight=peso)
    caminho = nx.astar_path(G, origem, destino, heuristic=h, weight=peso)
    return custo, caminho

# Função auxiliar para comparação de benchmark
def dijkstra_heap(G, origem, destino, peso="length"):
    return nx.bidirectional_dijkstra(G, origem, destino, weight=peso)

print("Funções A* e Dijkstra definidas.")

Funções A* e Dijkstra definidas.


In [6]:
import math
import osmnx as ox

VELOCIDADE_PEDESTRE = 1.4  # m/s

# ── Avaliação de um candidato P ──────────────────────────────────────────────
def avaliar_P_astar(G, no_P, no_B, dist_walk, peso_carro, peso_label):
    """Calcula distância + tempo total (caminhada A→P e carro P→B) com A*."""
    carro_dist,  _ = busca_astar(G, no_P, no_B, peso="length")
    carro_tempo, _ = busca_astar(G, no_P, no_B, peso=peso_carro)
    tempo_walk     = dist_walk / VELOCIDADE_PEDESTRE
    return {
        "no_P":          no_P,
        "walk_m":        dist_walk,
        "dist_total_m":  dist_walk + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
    }

def melhor_P_astar(G, candidatos, no_B, criterio, peso_carro="length", peso_label="dist"):
    """Varre todos os candidatos P e retorna o que minimiza o critério dado."""
    melhor_custo, melhor_info = math.inf, None
    for no_P, dist_walk in candidatos.items():
        if no_P not in G.nodes:
            continue
        info = avaliar_P_astar(G, no_P, no_B, dist_walk, peso_carro, peso_label)
        if info[criterio] < melhor_custo:
            melhor_custo = info[criterio]
            melhor_info  = info
    return melhor_info

# ── Cenário 4: sem caminhada (parte de A ou do P mais próximo) ───────────────
def cenario4_sem_caminhada_astar(G, G_walk, no_A_walk, no_A_drive, candidatos, no_B):
    """
    Cenário 4 — parte do ponto A exato se possível, sem impor caminhada.
    Se A não estiver no grafo drive, usa o candidato P mais próximo.
    """
    snap_lat = G.nodes[no_A_drive]["y"]
    snap_lon = G.nodes[no_A_drive]["x"]
    dist_snap = ox.distance.great_circle(
        PONTO_A["lat"], PONTO_A["lon"], snap_lat, snap_lon
    )
    TOLERANCIA_M = 1.0

    if dist_snap < TOLERANCIA_M:
        no_P, walk_m, caminhou = no_A_drive, 0.0, False
    else:
        no_P_mais_proximo, menor_dist = None, math.inf
        for no_P_cand, dist_walk_cand in candidatos.items():
            if no_P_cand in G.nodes and dist_walk_cand < menor_dist:
                menor_dist, no_P_mais_proximo = dist_walk_cand, no_P_cand
        if no_P_mais_proximo is None:
            no_P, walk_m, caminhou = no_A_drive, dist_snap, True
        else:
            no_P, walk_m, caminhou = no_P_mais_proximo, menor_dist, True

    carro_dist,  _ = busca_astar(G, no_P, no_B, peso="length")
    carro_tempo, _ = busca_astar(G, no_P, no_B, peso="travel_time")
    tempo_walk     = walk_m / VELOCIDADE_PEDESTRE
    return {
        "no_P":          no_P,
        "walk_m":        walk_m,
        "dist_total_m":  walk_m + carro_dist,
        "tempo_total_s": tempo_walk + carro_tempo,
        "c4_caminhou":   caminhou,
    }

# ── Orquestração dos 5 cenários ──────────────────────────────────────────────
def calcular_5_cenarios_astar(G, G_tr, G_walk, candidatos, no_A_walk, no_A_drive, no_B):
    c1 = melhor_P_astar(G,    candidatos, no_B, "dist_total_m",  "length",               "dist")
    c2 = melhor_P_astar(G,    candidatos, no_B, "tempo_total_s", "travel_time",          "time")
    c3 = melhor_P_astar(G_tr, candidatos, no_B, "tempo_total_s", "travel_time_transito", "time")
    c4 = cenario4_sem_caminhada_astar(G, G_walk, no_A_walk, no_A_drive, candidatos, no_B)

    ganho_dist  = c4["dist_total_m"]  - c1["dist_total_m"]
    ganho_tempo = c4["tempo_total_s"] - c2["tempo_total_s"]

    label_c4 = (
        f"4 – Sem caminhada (A∉drive, walk {c4['walk_m']:.0f} m)"
        if c4["c4_caminhou"] else "4 – Sem caminhada (A→B direto)"
    )

    tabela = [
        ("1 – Menor distância (c/ P)",        c1["dist_total_m"], c1["tempo_total_s"]),
        ("2 – Mais rápida s/ trânsito (c/P)", c2["dist_total_m"], c2["tempo_total_s"]),
        ("3 – Mais rápida c/ trânsito (c/P)", c3["dist_total_m"], c3["tempo_total_s"]),
        (label_c4,                              c4["dist_total_m"], c4["tempo_total_s"]),
        ("5 – Ganho ao caminhar (vs cen.4)",  ganho_dist,          ganho_tempo),
    ]
    return tabela, (c1, c2, c3, c4)

def imprimir_cenarios(titulo, tabela):
    print(f"\n=== {titulo} ===")
    print(f"{'Cenário':<52} {'Distância (m)':>14} {'Tempo (s)':>12}")
    print("—" * 82)
    for label, dist, tempo in tabela:
        d = f"{dist:.1f}"  if isinstance(dist,  float) else dist
        t = f"{tempo:.1f}" if isinstance(tempo, float) else tempo
        print(f"{label:<52} {d:>14} {t:>12}")

# Execução
tabela_astar, (c1, c2, c3, c4) = calcular_5_cenarios_astar(
    G, G_tr, G_walk, candidatos, no_A_walk, no_A, no_B
)
imprimir_cenarios("A* com heurística geográfica (todos os candidatos)", tabela_astar)

print(f"\nP ótimo (distância): nó {c1['no_P']} — caminhada {c1['walk_m']:.0f} m")
print(f"P ótimo (tempo)    : nó {c2['no_P']} — caminhada {c2['walk_m']:.0f} m")
print(f"P ótimo (trânsito) : nó {c3['no_P']} — caminhada {c3['walk_m']:.0f} m")
if c4["c4_caminhou"]:
    print(f"C4 (sem caminhada) : A NÃO está no grafo drive → caminhou {c4['walk_m']:.0f} m até nó {c4['no_P']}")
else:
    print(f"C4 (sem caminhada) : A está no grafo drive → partiu direto do nó {c4['no_P']}")


=== A* com heurística geográfica (todos os candidatos) ===
Cenário                                               Distância (m)    Tempo (s)
——————————————————————————————————————————————————————————————————————————————————
1 – Menor distância (c/ P)                                   2990.1       2859.3
2 – Mais rápida s/ trânsito (c/P)                            3238.4        320.0
3 – Mais rápida c/ trânsito (c/P)                            3238.4        624.4
4 – Sem caminhada (A∉drive, walk 144 m)                      3238.4        320.0
5 – Ganho ao caminhar (vs cen.4)                              248.2          0.0

P ótimo (distância): nó 501834700 — caminhada 458 m
P ótimo (tempo)    : nó 500986659 — caminhada 144 m
P ótimo (trânsito) : nó 500986659 — caminhada 144 m
C4 (sem caminhada) : A NÃO está no grafo drive → caminhou 144 m até nó 500986659


In [7]:
import networkx as nx

nos_expandidos_dijkstra = [0]
nos_expandidos_astar = [0]

# Wrapper para contar nós expandidos no Dijkstra Bidirecional
def dijkstra_contado(G, origem, destino, weight="length"):
    contador = 0
    visitados = set()
    import heapq, math
    dist = {n: math.inf for n in G.nodes}
    dist[origem] = 0
    heap = [(0, origem)]
    while heap:
        d, u = heapq.heappop(heap)
        if u in visitados:
            continue
        visitados.add(u)
        contador += 1
        if u == destino:
            break
        for v, data in G[u].items():
            for edge in data.values():
                w = edge.get(weight, 1)
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    heapq.heappush(heap, (dist[v], v))
    nos_expandidos_dijkstra[0] = contador
    return dist[destino], []

# Wrapper para contar nós expandidos no A*
def astar_contado(G, origem, destino, peso="length"):
    contador = 0
    original_astar = nx.astar_path

    class ContadorVisitas:
        def __init__(self):
            self.visitados = set()

    cv = ContadorVisitas()
    h = lambda u, v: heuristica_geo(u, v, G, peso=peso)

    import heapq, math
    dist = {n: math.inf for n in G.nodes}
    dist[origem] = 0
    heap = [(h(origem, destino), 0, origem)]
    visitados = set()

    while heap:
        _, d, u = heapq.heappop(heap)
        if u in visitados:
            continue
        visitados.add(u)
        contador += 1
        if u == destino:
            break
        for v, data in G[u].items():
            for edge in data.values():
                w = edge.get(peso, 1)
                nd = d + w
                if nd < dist[v]:
                    dist[v] = nd
                    heapq.heappush(heap, (nd + h(v, destino), nd, v))

    nos_expandidos_astar[0] = contador
    return dist[destino], []

# Teste para um par A→B
peso = "length"
dijkstra_contado(G, no_A, no_B, weight=peso)
astar_contado(G, no_A, no_B, peso=peso)

print(f"Nós expandidos — Dijkstra Heap : {nos_expandidos_dijkstra[0]}")
print(f"Nós expandidos — A*            : {nos_expandidos_astar[0]}")
print(f"Redução: {(1 - nos_expandidos_astar[0]/nos_expandidos_dijkstra[0])*100:.1f}%")

Nós expandidos — Dijkstra Heap : 753
Nós expandidos — A*            : 228
Redução: 69.7%


In [8]:
import time
import pandas as pd

linhas = []
for peso, label in [("length", "Distância"), ("travel_time", "Tempo")]:
    # Dijkstra Bidirecional (referência)
    t0 = time.perf_counter()
    dijkstra_heap(G, no_A, no_B, peso)
    td = time.perf_counter() - t0

    # A* com heurística geográfica
    t0 = time.perf_counter()
    busca_astar(G, no_A, no_B, peso)
    ta = time.perf_counter() - t0

    linhas.append({"Peso": label, "Dijkstra (s)": td, "A* (s)": ta, "Speedup (D/A*)": td / ta})

df = pd.DataFrame(linhas).set_index("Peso")
print(df.to_string(float_format="{:.4f}".format))
print(f"\nGrafo: {G.number_of_nodes()} nós | {G.number_of_edges()} arestas")
print("A* é mais rápido por guiar a busca com a heurística geográfica (linha reta).")


           Dijkstra (s)  A* (s)  Speedup (D/A*)
Peso                                           
Distância        0.0061  0.0347          0.1759
Tempo            0.0130  0.0520          0.2501

Grafo: 4965 nós | 12359 arestas
A* é mais rápido por guiar a busca com a heurística geográfica (linha reta).


In [9]:
def diagnostico_cenarios(c1, c2, c3, no_A):
    print("=== Diagnóstico: P escolhido por critério ===\n")
    for label, c in [
        ("Cenário 1 (menor distância)",        c1),
        ("Cenário 2 (mais rápido s/ trânsito)", c2),
        ("Cenário 3 (mais rápido c/ trânsito)", c3),
    ]:
        eh_A = "← é o próprio A (sem caminhar)" if c["no_P"] == no_A else ""
        print(f"{label:<38} P = {c['no_P']:<12} caminhada = {c['walk_m']:>6.0f} m {eh_A}")

    print()
    if c1["no_P"] == c2["no_P"]:
        print(f"✓ Cenários 1 e 2 escolheram o MESMO P (nó {c1['no_P']}).")
        _, rota1 = busca_astar(G, c1["no_P"], no_B, peso="length")
        _, rota2 = busca_astar(G, c2["no_P"], no_B, peso="travel_time")
        if rota1 == rota2:
            print("✓ A rota P→B também é EXATAMENTE a mesma sequência de nós.")
        else:
            print(f"⚠ Mesmo P, mas rotas P→B são sequências DIFERENTES de nós.")
            print(f"  Rota por distância: {len(rota1)} nós | Rota por tempo: {len(rota2)} nós")
    else:
        print(f"✗ Cenários 1 e 2 escolheram P DIFERENTES ({c1['no_P']} vs {c2['no_P']}).")
        print("  → Esperado: o ponto de menor distância pode não ser o de menor tempo.")

diagnostico_cenarios(c1, c2, c3, no_A)


=== Diagnóstico: P escolhido por critério ===

Cenário 1 (menor distância)            P = 501834700    caminhada =    458 m 
Cenário 2 (mais rápido s/ trânsito)    P = 500986659    caminhada =    144 m 
Cenário 3 (mais rápido c/ trânsito)    P = 500986659    caminhada =    144 m 

✗ Cenários 1 e 2 escolheram P DIFERENTES (501834700 vs 500986659).
  → Esperado: o ponto de menor distância pode não ser o de menor tempo.


In [10]:
import folium

def coords_rota(G, caminho):
    return [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in caminho]

def montar_mapa_astar(c1, c2, c3, c4, no_A, no_B):
    """Monta o mapa Folium com os 4 cenários, usando A* para calcular as rotas."""    # Rotas de caminhada A→P (rede pedestre)
    _, rota_walk_c1 = busca_astar(G_walk, no_A_walk, c1["no_P"], peso="length")
    _, rota_walk_c2 = busca_astar(G_walk, no_A_walk, c2["no_P"], peso="length")
    _, rota_walk_c3 = busca_astar(G_walk, no_A_walk, c3["no_P"], peso="length")

    # Rotas de carro P→B (rede de drive)
    _, rota_carro_c1 = busca_astar(G,    c1["no_P"], no_B, peso="length")
    _, rota_carro_c2 = busca_astar(G,    c2["no_P"], no_B, peso="length")
    _, rota_carro_c3 = busca_astar(G_tr, c3["no_P"], no_B, peso="travel_time_transito")

    # Cenário 4
    _, rota_carro_c4 = busca_astar(G, c4["no_P"], no_B, peso="length")
    rota_walk_c4 = []
    if c4["c4_caminhou"]:
        _, rota_walk_c4 = busca_astar(G_walk, no_A_walk, c4["no_P"], peso="length")

    mapa = folium.Map(location=[centro[0], centro[1]], zoom_start=14, tiles="CartoDB positron")

    # Marcadores A e B
    folium.Marker([PONTO_A["lat"], PONTO_A["lon"]], popup=f"A – {PONTO_A['nome']}",
                  icon=folium.Icon(color="green", icon="home")).add_to(mapa)
    folium.Marker([PONTO_B["lat"], PONTO_B["lon"]], popup=f"B – {PONTO_B['nome']}",
                  icon=folium.Icon(color="red",   icon="flag")).add_to(mapa)

    # Marcadores P de cada cenário
    for c_info, cor, rotulo in [(c1, "blue", "P¹ – Menor distância"),
                                  (c2, "purple", "P² – Mais rápido s/ trânsito"),
                                  (c3, "orange", "P³ – Mais rápido c/ trânsito")]:
        folium.Marker(
            [G.nodes[c_info["no_P"]]["y"], G.nodes[c_info["no_P"]]["x"]],
            popup=f"{rotulo}<br>{c_info['walk_m']:.0f} m de caminhada",
            icon=folium.Icon(color=cor, icon="car")
        ).add_to(mapa)

    if c4["c4_caminhou"] and c4["no_P"] not in (c1["no_P"], c2["no_P"], c3["no_P"]):
        folium.Marker(
            [G.nodes[c4["no_P"]]["y"], G.nodes[c4["no_P"]]["x"]],
            popup=f"P⁴ – C4 (P mais próximo)<br>{c4['walk_m']:.0f} m",
            icon=folium.Icon(color="gray", icon="car")
        ).add_to(mapa)

    # Raio de caminhada
    folium.Circle([PONTO_A["lat"], PONTO_A["lon"]], radius=X,
                  color="gray", fill=True, fill_opacity=0.07,
                  tooltip=f"Raio {X} m").add_to(mapa)

    # ── Feature Groups (camadas toggle) ─────────────────────────────────────
    for c_info, G_c, rota_walk, rota_carro, cor, nome_camada in [
        (c1, G,    rota_walk_c1, rota_carro_c1, "blue",   "C1 – Menor distância"),
        (c2, G,    rota_walk_c2, rota_carro_c2, "purple", "C2 – Mais rápido s/ trânsito"),
        (c3, G_tr, rota_walk_c3, rota_carro_c3, "orange", "C3 – Mais rápido c/ trânsito"),
    ]:
        fg = folium.FeatureGroup(name=nome_camada, show=True)
        folium.PolyLine(coords_rota(G_walk, rota_walk), color=cor, weight=4,
                        tooltip=f"{nome_camada} – Caminhada A→P").add_to(fg)
        folium.PolyLine(coords_rota(G_c, rota_carro),   color=cor, weight=4, dash_array="8",
                        tooltip=f"{nome_camada} – Carro P→B").add_to(fg)
        fg.add_to(mapa)

    label_c4 = (
        f"C4 – Sem caminhada (walk forçado {c4['walk_m']:.0f} m)" if c4["c4_caminhou"]
        else "C4 – Sem caminhada (A→B direto)"
    )
    fg4 = folium.FeatureGroup(name=label_c4, show=False)
    if c4["c4_caminhou"] and rota_walk_c4:
        folium.PolyLine(coords_rota(G_walk, rota_walk_c4), color="gray", weight=3,
                        opacity=0.7, tooltip="C4 – Caminhada forçada A→P⁴").add_to(fg4)
    folium.PolyLine(coords_rota(G, rota_carro_c4), color="gray", weight=3, opacity=0.7,
                    dash_array="8", tooltip="C4 – Carro P→B").add_to(fg4)
    fg4.add_to(mapa)

    folium.map.Marker(
        [centro[0] + 0.01, centro[1]],
        icon=folium.DivIcon(html='<div style="font-size:16px; font-weight:bold; background:white; padding:4px 8px; border-radius:4px;">Roteamento A*</div>')
    ).add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)
    return mapa

mapa_astar = montar_mapa_astar(c1, c2, c3, c4, no_A, no_B)
mapa_astar
